# AI Engineering Interview Prep
## Section: Python

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 651+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (30 Qs)",
    "18. FastAPI (30 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 🐍 Section 17 — Python

### Q1. How do generators and `yield` work? How are they useful in AI applications?
**A:** Standard functions run to completion and return a single result. A generator function uses `yield` to return a value, suspend its execution, and save its state. When called again, it resumes right where it left off. In AI engineering, memory is a major constraint. If you're processing a corpus of 10 million documents for RAG, loading them all into a Python list will crash your container (OOM). With a generator, you stream documents from disk or DB one by one, embed them in batches, and write to the vector DB, keeping memory footprint minimal.

### Q2. How do decorators work in Python? How do you write a decorator that accepts arguments?
**A:** Decorators are wrappers around functions (or classes) that allow you to execute code before and after the wrapped function runs, without modifying its source. To write a decorator that accepts arguments (like a retry decorator with a `max_retries` count), you need three levels of functions:
1. The outer function accepts the decorator arguments.
2. The middle function (the actual decorator) accepts the target function.
3. The inner function (the wrapper) accepts the target function's arguments, runs the logic (e.g., a try-except loop up to `max_retries`), and returns the result.
Using `@functools.wraps` is essential to preserve the original function's name and docstring.

### Q3. Explain the context manager protocol and how it applies to resource management?
**A:** Context managers ensure resources are cleaned up properly (like files, database sessions, or GPU memory), even if exceptions occur. You use them with the `with` statement. Under the hood, Python calls `__enter__` to set up the resource and `__exit__` to clean it up when the block finishes. You can also write them easily using the `@contextmanager` decorator from `contextlib` with a `try/finally` block. In AI, they are vital for managing PyTorch GPU sessions (`with torch.no_grad():`) to disable gradient calculations and free up VRAM, or database connections in RAG pipelines.

### Q4. What is the GIL (Global Interpreter Lock)? How does it affect multi-threading vs. multi-processing vs. asyncio?
**A:** The GIL is a mutex in CPython that ensures only one thread executes Python bytecode at a time. This prevents race conditions in memory management but makes multi-threading useless for CPU-bound tasks (like calculating embeddings or matrix multiplications in Python), as they won't run on multiple CPU cores.
- **For CPU-bound tasks:** Use `multiprocessing` to spawn separate OS processes with their own Python interpreters and memory spaces, bypassing the GIL.
- **For I/O-bound tasks (like LLM API calls, fetching web pages):** Use `asyncio` or `threading`. The thread/task yields execution while waiting for the network, letting other threads run.

### Q5. How does `asyncio` achieve concurrency under the hood?
**A:** `asyncio` achieves concurrency using cooperative multitasking driven by an event loop. Unlike OS-level multi-threading (where the OS preemptively switches threads), `asyncio` runs in a single thread. When a function hits an `await` statement (like a database query or an LLM call), it yields control back to the event loop, which immediately schedules another task that is ready to run. This is incredibly lightweight. Spawning 10,000 OS threads will crash your server, but spawning 10,000 asyncio tasks uses minimal memory because there is no thread context-switching overhead.

### Q6. What are metaclasses, and how do they differ from class decorators?
**A:** A class decorator takes a fully constructed class and modifies it. A metaclass is a "class of a class"—it defines *how* a class is constructed in the first place (it overrides `__new__` or `__init__` of the `type` class). You use class decorators for simple post-construction modifications. You use metaclasses when you need deep, structural control over class creation, inheritance, or field registration. For example, Pydantic uses metaclasses to parse field annotations and compile validation schemas before the class is ever instantiated.

### Q7. How does Python manage memory, and how do you prevent leaks in RAG systems?
**A:** Python uses reference counting (deleting objects when their reference count hits 0) combined with a generational Garbage Collector to catch cyclic references (e.g., Object A references B, and B references A). In RAG systems, memory leaks often happen when storing massive document chunks in long-lived variables (like global lists, caches, or state variables in class instances). To prevent this:
1. Avoid global state.
2. Use generator streams to process data.
3. Explicitly clear large data structures (`del my_list` or `my_list.clear()`).
4. Use weak references (`weakref`) if you need to keep references without increasing the reference count.

### Q8. Explain dunder (magic) methods and when you would implement them.
**A:** Dunder (double underscore) methods allow custom classes to hook into Python's built-in syntax.
- `__call__`: Makes an instance callable like a function (e.g., LangChain's Runnables implement `__call__` or `invoke`).
- `__getitem__` / `__setitem__`: Allows indexing like a list or dictionary (`my_obj[key]`).
- `__getattr__` / `__setattr__`: Customizes attribute access.
- `__enter__` / `__exit__`: Implements the context manager protocol.
You implement them to build clean, intuitive APIs for other developers in your team.

### Q9. Why is type hinting critical in AI codebases? What are Annotated, Union, and Generic?
**A:** LLMs return unpredictable structures, and RAG pipelines pass around complex objects. Without type hints, debugging runtime type errors is a nightmare. Using `mypy` helps catch errors during CI/CD before code runs.
- `Annotated`: Attaches metadata to types (e.g., `Annotated[str, "metadata info"]` used in LangGraph to define reducers like `Annotated[list, add_messages]`).
- `Union` (or `|` in Python 3.10+): Specifies that a value can be one of multiple types.
- `Generic`: Allows you to write reusable container classes (like a custom `Response[T]` class) that maintain type checking regardless of the wrapped object's type.

### Q10. Compare Python dataclasses vs. Pydantic models. When do you use each?
**A:** - **Dataclasses (standard library):** Great for internal data containers. They generate boilerplate (`__init__`, `__repr__`) automatically. They do *not* perform runtime validation by default; they only document types. They are lightweight and fast.
- **Pydantic Models:** Perform strict runtime data validation and serialization. If a field type doesn't match the schema, it raises a validation error. Essential for API payloads (FastAPI), parsing structured outputs from LLMs, and validating complex configurations.
**Rule:** Use dataclasses for internal, trusted data structures. Use Pydantic for system boundaries (APIs, LLM outputs, user input).

### Q11. What is the difference between `is` and `==`? Under what conditions can `is` behave unexpectedly?
**A:** `==` checks whether the values of two objects are equal, whereas `is` checks whether they point to the exact same memory address (reference identity). `is` can behave unexpectedly due to CPython's memory optimization strategies, such as caching small integers (from -5 to 256) and short strings (interning). In these cases, two variables initialized with the same values may refer to the same pre-allocated object, making `a is b` evaluate to `True`. Always use `==` for comparing values.

### Q12. Why are mutable default arguments dangerous in Python, and how do you fix them?
**A:** Default arguments are evaluated once at function definition time, not at execution time. If you use a mutable object (like a list `[]` or dict `{}`) as a default argument, all function invocations that omit this parameter will share the same object instance. If one call mutates it, all subsequent calls will see the modified object, introducing silent bugs. The standard fix is to use `None` as the default value and initialize the mutable object inside the function body: `if my_param is None: my_param = []`.

### Q13. What is the output of `list_val = [[0]] * 3`? Explain why modifying `list_val[0][0]` changes other elements.
**A:** The output of `[[0]] * 3` is `[[0], [0], [0]]`. However, modifying `list_val[0][0] = 5` results in `[[5], [5], [5]]`. This occurs because the multiplication operator `*` does not create three independent sublists. Instead, it copies the *reference* to the single inner list `[0]` three times. Thus, all three elements of the outer list point to the exact same memory address, and any in-place mutation to one sublist is reflected across all of them.

### Q14. Why can `x = x + [1]` behave differently than `x += [1]` in Python lists?
**A:** - `x = x + [1]` creates a new list in memory by concatenating `x` and `[1]`, and re-binds the variable `x` to this new object (out-of-place operation).
- `x += [1]` calls the in-place addition method `x.__iadd__([1])` (equivalent to `x.extend([1])`), modifying the existing list object directly. If multiple variables reference the same list object, using `+=` will affect all of them, while `+` will only update the variable being assigned to.

### Q15. Explain late binding in Python closures and how it can cause unexpected outputs inside loops.
**A:** Python closures bind variables by reference, not by value. The variables are looked up when the inner function is called (late binding), rather than when it is defined. If you define nested functions (or lambda expressions) inside a loop, they all reference the same loop variable. By the time they are called, the loop has completed, and the variable holds its final value. To capture the variable's value at definition time, use a default argument: `lambda x, i=i: x * i`.

### Q16. What is the difference between a shallow copy and a deep copy? When is each necessary?
**A:** - **Shallow Copy (`copy.copy()`):** Creates a new container object, but inserts references to the nested objects from the original container. Modifying a nested object affects both copies.
- **Deep Copy (`copy.deepcopy()`):** Recursively creates a new container and duplicates all nested objects. It is necessary when you want to duplicate a nested data structure (like a list of dictionaries or an agent's configuration state) and ensure that mutations on the clone do not corrupt the original state.

### Q17. Why can a `finally` block override a `return` statement in Python?
**A:** The `finally` block is guaranteed to execute before the `try` block exits, including when a `return` is reached. If the `finally` block itself contains a `return` or a `raise` statement, the exception or return value from the `try` or `except` blocks is discarded, and the value from the `finally` block is returned instead. This can lead to hard-to-debug behaviors if return statements are misplaced in clean-up code.

### Q18. Explain the difference between class variables and instance variables. How can dynamic mutation cause bugs?
**A:** - **Class Variables:** Defined inside the class body but outside methods. They are shared by all instances of the class.
- **Instance Variables:** Defined inside methods (typically `__init__`) using `self`. They are unique to each instance.
**Bugs:** If you mutate a class variable using the class name (`MyClass.var = value`), it changes for all instances. However, if you attempt to mutate it using an instance (`instance.var = value`), Python silently creates a new instance variable of the same name, shadowing the class variable for that instance only.

### Q19. What is name mangling in Python, and why does it matter?
**A:** Name mangling is a mechanism where any attribute prefixed with two leading underscores (e.g. `__my_method`) is renamed to `_ClassName__my_method` at compile time. It is not designed to enforce privacy or access control. Rather, it prevents attribute name clashes in subclasses when a class is extended, ensuring that internal methods or variables are not accidentally overridden by subclasses.

### Q20. What is the difference between `dict.get(key)` and indexing `dict[key]`? When should you use each?
**A:** - `dict[key]` performs a direct lookup and raises a `KeyError` if the key does not exist. Use this when the key is expected to be present, and its absence indicates an exceptional state.
- `dict.get(key, default=None)` returns the value if the key exists, or the `default` value otherwise, without throwing an error. Use this when handling optional fields or external payloads where keys might be missing.

### Q21. Explain how arguments are passed in Python. Is it pass-by-value or pass-by-reference?
**A:** Python uses **pass-by-object-reference** (also called pass-by-assignment). When you pass an argument to a function, Python binds the local parameter name to the exact same object in memory.
- If the object is **mutable** (like a list or dict), changes made to the object inside the function (like `append`) will be visible to the caller.
- If the object is **immutable** (like an int, string, or tuple), you cannot mutate it. Any re-assignment inside the function merely binds the local name to a new object, leaving the caller's variable unchanged.

### Q22. What are Python namespaces, and what is the LEGB rule for scope resolution?
**A:** A namespace is a dictionary mapping names to objects. Python has built-in, global, enclosing (in closures), and local namespaces. When you reference a variable, Python searches these scopes in a strict order defined by the **LEGB rule**:
1. **Local (L):** Inside the current function.
2. **Enclosing (E):** Inside any outer enclosing functions (closures).
3. **Global (G):** At the module level.
4. **Built-in (B):** Python's built-in names (like `len`, `int`).
If the name is not found in any scope, a `NameError` is raised.

### Q23. What is Method Resolution Order (MRO) in Python, and how is it calculated for multiple inheritance?
**A:** Method Resolution Order (MRO) defines the exact sequence in which Python searches for attributes or methods in a class hierarchy, especially under multiple inheritance. Since Python 2.3, this is calculated using the **C3 Linearization** algorithm. This algorithm guarantees two properties: subclass precedence (a subclass is always searched before its parent classes) and monotonicity (the order of base classes declared is preserved). You can check a class's MRO using the `__mro__` attribute (e.g. `MyClass.__mro__`).

### Q24. What are descriptors in Python, and how do they enable properties (`@property`) under the hood?
**A:** A descriptor is any object that implements at least one of the protocol methods: `__get__()`, `__set__()`, or `__delete__()` in its class. When you define an attribute on a class that is a descriptor, accessing or modifying that attribute on an instance automatically routes the access to the descriptor's methods. The `@property` decorator is implemented as a built-in descriptor that intercepts attribute lookups and calls the corresponding getter, setter, and deleter methods.

### Q25. What are `__slots__` in Python classes, and how do they improve performance and memory usage?
**A:** By default, Python instances store their attributes in a dynamic dictionary (`__dict__`), which has significant memory overhead due to hash tables. By defining `__slots__ = ('attr1', 'attr2')` in a class, you tell Python to allocate space for a fixed set of attributes in a lightweight array structure instead. This restricts instances from having dynamic attributes, but it reduces memory usage (often by 50-70%) and speeds up attribute lookup. It is ideal for classes with millions of active instances.

### Q26. What is the difference between `__new__` and `__init__` methods in class instantiation?
**A:** - `__new__(cls, *args, **kwargs)` is a static constructor method responsible for *creating* and returning a new instance of the class. It is the first method called during instantiation.
- `__init__(self, *args, **kwargs)` is an initializer method that configures the newly created instance (`self`). It does not return any value.
Override `__new__` when you need to subclass immutable types (like subclassing `tuple` or `str`), customize object creation, or implement design patterns like the Singleton pattern.

### Q27. How does NumPy leverage vectorization and memory layouts (C-contiguous vs Fortran-contiguous) for speed?
**A:** Vectorization replaces Python loops with fast, compiled C code utilizing SIMD instructions. For optimal performance, NumPy arrays rely on contiguous memory blocks:
- **C-contiguous (row-major):** Elements in a row are contiguous. Fastest when iterating along rows (default layout).
- **Fortran-contiguous (column-major):** Elements in a column are contiguous. Fastest when iterating along columns.
Matching your loop iteration direction with the memory layout prevents CPU cache misses and achieves maximum execution speed.

### Q28. How are NumPy arrays advantageous over Python lists? Explain in terms of memory layout and CPU caching.
**A:** - **Memory Layout:** NumPy arrays are homogeneous (same data type) and stored as contiguous blocks of raw memory. Python lists are arrays of pointers pointing to scattered objects in memory (causing memory indirection).
- **CPU Cache:** Contiguous memory layout allows the CPU to prefetch next elements into L1/L2 cache (cache locality). Lists cause random access patterns, causing CPU cache misses.
- **Overhead:** NumPy performs operations in compiled C, avoiding Python's runtime type-checking and method dispatch overhead.

### Q29. How does Pandas handle missing values under the hood (e.g. NaN, None), and how do you handle them memory-safely?
**A:** Pandas traditionally uses floating-point `NaN` (Not a Number) to represent missing values, which coerces integer columns to float. Modern Pandas (v1.0+) introduces nullable types (e.g. `Int64`, `boolean`) that keep integer representation using an internal boolean mask. To handle them memory-safely:
- Avoid creating intermediate copies of dataframes; perform cleaning using `inplace=True` or inline operations.
- Use `df.isna()` to detect and filter, and `df.fillna()` or `df.dropna()` to clean data before feeding it into down-stream RAG or ML models.

### Q30. What is the difference between a module and a package in Python? How does relative import syntax work?
**A:** - **Module:** A single `.py` file containing classes, functions, and variables.
- **Package:** A directory containing modules and an `__init__.py` file (which registers the directory as a package).
- **Relative Imports:** Import sibling modules using dot notation (`from . import sibling` or `from ..parent import uncle`). They are resolved based on the module's `__name__` attribute. Relative imports only work within a package execution context; running a module with relative imports as a standalone script results in an `ImportError`.